In [ ]:
print("🚀")

In [ ]:
! uv pip install langgraph-checkpoint-postgres psycopg[binary]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHORT-TERM MEMORY — MINIMAL CHECKPOINTER SETUP
# ══════════════════════════════════════════════════════════════════════════════
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig 
from dotenv import load_dotenv
load_dotenv()

# ── Tool definition ────────────────────────────────────────────────────────────
def lookup_orders() -> str:
    """Fetch recent orders for the current user."""
    return "Order #4821: delivered. Order #4822: in transit."

# ── Agent with checkpointer ────────────────────────────────────────────────────
agent = create_agent(
    model="anthropic:claude-sonnet-4-6",   # provider:model-name string
    tools=[lookup_orders],
    checkpointer=InMemorySaver(),           # enables thread-scoped memory
)

# ── Thread config — same thread_id restores the same state ─────────────────────
thread_config:RunnableConfig = {"configurable": {"thread_id": "user-priya-session-1"}}

# Turn 1
r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, I'm Priya. Check my orders."}]},
    thread_config,
)["messages"][-1].content
print(r1)  # mentions order #4821 and #4822

# Turn 2 — agent still knows the name "Priya" because state was checkpointed
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    thread_config,
)["messages"][-1].content
print(r2)  # "Your name is Priya."